In [1]:
struct Pentomino
    id::Int
    name::Char
    tiles::Vector{Tuple{Int, Int}}  # list of (x, y) coordinates
end

# Represents one pentomino being used with a specific input edge (connected to the last pentomino) and one output edge (will be connected to the next pentomino) 
struct Possibility
    new_orient::Int  # orientation of output edge
    area_change_8x::Int  # change to area by the shoelace formula, we save 8x fold the value to have it as an integer
    pos_diff::Tuple{Int, Int}  # change in position, so difference between output and input edge coordinates

    # data for reconstructing the pentomino placement
    pento_name::Char  # name of pentomino that is used
    shifted_tiles::Vector{Tuple{Int, Int}}  # tiles of the pentomino shifted such that the tile adjacent to input edge is at (0, 0)
    next_tile_offset::Tuple{Int, Int}  # coordinate of tile from next pentomino which will be adjacent to output edge in same reference frame as shifted_tiles
end

In [2]:
# the 12 pentominoes before any normalization
const raw_pentos = [
    Pentomino(1, 'F', [(0, 1), (1, 1), (1, 0), (1, 2), (2, 0)]),
    Pentomino(2, 'I', [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4)]),
    Pentomino(3, 'L', [(0, 0), (0, 1), (0, 2), (0, 3), (1, 0)]),
    Pentomino(4, 'N', [(0, 0), (0, 1), (1, 1), (1, 2), (1, 3)]),
    Pentomino(5, 'P', [(0, 0), (0, 1), (1, 0), (1, 1), (0, 2)]),
    Pentomino(6, 'T', [(0, 0), (1, 0), (2, 0), (1, 1), (1, 2)]),
    Pentomino(7, 'U', [(0, 0), (0, 1), (1, 0), (2, 0), (2, 1)]),
    Pentomino(8, 'V', [(0, 0), (0, 1), (0, 2), (1, 0), (2, 0)]),
    Pentomino(9, 'W', [(0, 0), (0, 1), (1, 1), (1, 2), (2, 2)]),
    Pentomino(10, 'X', [(1, 0), (0, 1), (1, 1), (2, 1), (1, 2)]),
    Pentomino(11, 'Y', [(0, 0), (0, 1), (0, 2), (0, 3), (1, 1)]),
    Pentomino(12, 'Z', [(0, 0), (1, 0), (1, 1), (1, 2), (2, 2)])
]

12-element Vector{Pentomino}:
 Pentomino(1, 'F', [(0, 1), (1, 1), (1, 0), (1, 2), (2, 0)])
 Pentomino(2, 'I', [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4)])
 Pentomino(3, 'L', [(0, 0), (0, 1), (0, 2), (0, 3), (1, 0)])
 Pentomino(4, 'N', [(0, 0), (0, 1), (1, 1), (1, 2), (1, 3)])
 Pentomino(5, 'P', [(0, 0), (0, 1), (1, 0), (1, 1), (0, 2)])
 Pentomino(6, 'T', [(0, 0), (1, 0), (2, 0), (1, 1), (1, 2)])
 Pentomino(7, 'U', [(0, 0), (0, 1), (1, 0), (2, 0), (2, 1)])
 Pentomino(8, 'V', [(0, 0), (0, 1), (0, 2), (1, 0), (2, 0)])
 Pentomino(9, 'W', [(0, 0), (0, 1), (1, 1), (1, 2), (2, 2)])
 Pentomino(10, 'X', [(1, 0), (0, 1), (1, 1), (2, 1), (1, 2)])
 Pentomino(11, 'Y', [(0, 0), (0, 1), (0, 2), (0, 3), (1, 1)])
 Pentomino(12, 'Z', [(0, 0), (1, 0), (1, 1), (1, 2), (2, 2)])

In [3]:
# normalize the tile set of the pentomino by shifting the minimum coordinates to 0
# finally tiles are also sorted in lexicographic order, which is not strictly necessary but done for consistency
function normalize(P)
    vals_x = [x for (x, y) in P]
    vals_y = [y for (x, y) in P]
    min_x = minimum(vals_x)
    min_y = minimum(vals_y)
    
    return sort([(x - min_x, y - min_y) for (x, y) in P])  # sort lexicographically
end

function rotate(P, num_rot)
    # transformation directly derived from standard rotation matrices (https://en.wikipedia.org/wiki/Rotation_matrix#Common_2D_rotations)

    if num_rot == 0  # no rotation
        return P
    elseif num_rot == 1  # rotation 90° counterclockwise (ccw)
        return [(-y, x) for (x, y) in P]
    elseif num_rot == 2  # rotation 180° ccw
        return [(-x, -y) for (x, y) in P]
    else  # rotation 270° ccw (same as 90° clockwise)
        return [(y, -x) for (x, y) in P]
    end
end

function flip(P)
    return [(y, x) for (x, y) in P]  # flip across 45° diagonal from bottom left to top right
end

# get all unique orientations of the 8 orientation (4 rotations + 4 flipped rotations) for each pentomino
function get_orientations(P)
    base = [normalize(rotate(P, r)) for r in 0:3]
    flipped = [normalize(rotate(flip(P), r)) for r in 0:3]
    return unique(vcat(base, flipped))
end

get_orientations (generic function with 1 method)

In [4]:
# Normalize all given pentominoes
const pentos = [Pentomino(p.id, p.name, normalize(p.tiles)) for p in raw_pentos]

12-element Vector{Pentomino}:
 Pentomino(1, 'F', [(0, 1), (1, 0), (1, 1), (1, 2), (2, 0)])
 Pentomino(2, 'I', [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4)])
 Pentomino(3, 'L', [(0, 0), (0, 1), (0, 2), (0, 3), (1, 0)])
 Pentomino(4, 'N', [(0, 0), (0, 1), (1, 1), (1, 2), (1, 3)])
 Pentomino(5, 'P', [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1)])
 Pentomino(6, 'T', [(0, 0), (1, 0), (1, 1), (1, 2), (2, 0)])
 Pentomino(7, 'U', [(0, 0), (0, 1), (1, 0), (2, 0), (2, 1)])
 Pentomino(8, 'V', [(0, 0), (0, 1), (0, 2), (1, 0), (2, 0)])
 Pentomino(9, 'W', [(0, 0), (0, 1), (1, 1), (1, 2), (2, 2)])
 Pentomino(10, 'X', [(0, 1), (1, 0), (1, 1), (1, 2), (2, 1)])
 Pentomino(11, 'Y', [(0, 0), (0, 1), (0, 2), (0, 3), (1, 1)])
 Pentomino(12, 'Z', [(0, 0), (1, 0), (1, 1), (1, 2), (2, 2)])

In [5]:
function pretty_print(P)
    P = normalize(P) # Normalize before drawing
    xs = first.(P)
    ys = last.(P)

    xmax = maximum(xs)
    ymax = maximum(ys)

    grid = fill('.', ymax + 1, xmax + 1)  # range from 0 to max, so we need max + 1 much space in each dimension

    for (x,y) in P
        grid[ymax - y + 1, x + 1] = '#'  # print in cartesian coordinates
    end

    # Print row by row
    for row in eachrow(grid)
        println(join(row))
    end
end

for p in pentos
    orients = get_orientations(p.tiles)

    for (i, tiles) in enumerate(orients)
        println("Pentomino $(p.name), Orientation $i:")
        pretty_print(tiles)
        println()  # extra visual spacing before next orientation
    end
end

Pentomino F, Orientation 1:
.#.
##.
.##

Pentomino F, Orientation 2:
..#
###
.#.

Pentomino F, Orientation 3:
##.
.##
.#.

Pentomino F, Orientation 4:
.#.
###
#..

Pentomino F, Orientation 5:
#..
###
.#.

Pentomino F, Orientation 6:
.#.
.##
##.

Pentomino F, Orientation 7:
.#.
###
..#

Pentomino F, Orientation 8:
.##
##.
.#.

Pentomino I, Orientation 1:
#
#
#
#
#

Pentomino I, Orientation 2:
#####

Pentomino L, Orientation 1:
#.
#.
#.
##

Pentomino L, Orientation 2:
...#
####

Pentomino L, Orientation 3:
##
.#
.#
.#

Pentomino L, Orientation 4:
####
#...

Pentomino L, Orientation 5:
#...
####

Pentomino L, Orientation 6:
.#
.#
.#
##

Pentomino L, Orientation 7:
####
...#

Pentomino L, Orientation 8:
##
#.
#.
#.

Pentomino N, Orientation 1:
.#
.#
##
#.

Pentomino N, Orientation 2:
###.
..##

Pentomino N, Orientation 3:
.#
##
#.
#.

Pentomino N, Orientation 4:
##..
.###

Pentomino N, Orientation 5:
.###
##..

Pentomino N, Orientation 6:
#.
#.
##
.#

Pentomino N, Orientation 7:
..##
###.


In [6]:
const DIRECTIONS = [(0, -1), (1, 0), (0, 1), (-1, 0)]

function compute_boundary(P)
    boundary_successor = Dict{Tuple{Int, Int}, Tuple{Int, Int}}()  # key: coordinates of starting point, value: coordinates of ending point
    edge_to_tile = Dict{Tuple{Tuple{Int, Int}, Tuple{Int, Int}}, Tuple{Tuple{Int, Int}, Tuple{Int, Int}}}()  # (perimeter edge) -> (adjacent inner tile, adjacent outer tile)

    for (x, y) in P
        for (dx, dy) in DIRECTIONS
            xx, yy = x + dx, y + dy
            if !((xx, yy) in P)  # pentomino tile (x, y) shares outside edge with non-pentomino tile (xx, yy)
                
                # (dx, dy) is a vector pointing from (x, y) to (xx, yy)
                # then (-dy, dx) is a vector rotated 90 deg counterclockwise (since (-dy, dx) = M * (dx, dy) where M is the 90 deg ccw rotation from here: https://en.wikipedia.org/wiki/Rotation_matrix#Common_2D_rotations)
                perpx2, perpy2 = -dy, dx  # half length to allow easier start and midpoint creation
                
                midx2, midy2 = x + xx, y + yy  # midpoint of the edge
                
                # calculate start and end points of current edge
                start_pt = (midx2 + perpx2, midy2 + perpy2)
                end_pt = (midx2 - perpx2, midy2 - perpy2)
                
                # store the clockwise successor relationship
                @assert !haskey(boundary_successor, start_pt)  # check that each point is only used once in the border (TODO: Heptomino counter example)
                boundary_successor[start_pt] = end_pt

                edge_to_tile[(start_pt, end_pt)] = ((x, y), (xx, yy))  # only used for reconstruction
            end
        end
    end

    return boundary_successor, edge_to_tile
end

println(compute_boundary(pentos[8].tiles))

(Dict((3, 1) => (5, 1), (1, 3) => (1, 1), (-1, 3) => (-1, 5), (5, -1) => (3, -1), (1, 5) => (1, 3), (1, -1) => (-1, -1), (-1, 5) => (1, 5), (5, 1) => (5, -1), (-1, -1) => (-1, 1), (3, -1) => (1, -1), (1, 1) => (3, 1), (-1, 1) => (-1, 3)), Dict(((-1, -1), (-1, 1)) => ((0, 0), (-1, 0)), ((-1, 5), (1, 5)) => ((0, 2), (0, 3)), ((1, -1), (-1, -1)) => ((0, 0), (0, -1)), ((5, -1), (3, -1)) => ((2, 0), (2, -1)), ((1, 1), (3, 1)) => ((1, 0), (1, 1)), ((-1, 3), (-1, 5)) => ((0, 2), (-1, 2)), ((3, 1), (5, 1)) => ((2, 0), (2, 1)), ((5, 1), (5, -1)) => ((2, 0), (3, 0)), ((1, 3), (1, 1)) => ((0, 1), (1, 1)), ((1, 5), (1, 3)) => ((0, 2), (1, 2)), ((3, -1), (1, -1)) => ((1, 0), (1, -1)), ((-1, 1), (-1, 3)) => ((0, 1), (-1, 1))))


In [7]:
function construct_boundary_path(P)
    boundary_successor, edge_to_tile = compute_boundary(P)
    
    # construct clockwise edge path as discussed
    start_pt = current = minimum(keys(boundary_successor))  # can be any edge, but for debugging choose deterministically
    path = [start_pt]
    
    current = boundary_successor[current]
    while current != start_pt
        push!(path, current)
        current = boundary_successor[current]
    end
    
    @assert length(path) == length(boundary_successor)
    
    return path, edge_to_tile
end

println(construct_boundary_path(pentos[8].tiles))

([(-1, -1), (-1, 1), (-1, 3), (-1, 5), (1, 5), (1, 3), (1, 1), (3, 1), (5, 1), (5, -1), (3, -1), (1, -1)], Dict(((-1, -1), (-1, 1)) => ((0, 0), (-1, 0)), ((-1, 5), (1, 5)) => ((0, 2), (0, 3)), ((1, -1), (-1, -1)) => ((0, 0), (0, -1)), ((5, -1), (3, -1)) => ((2, 0), (2, -1)), ((1, 1), (3, 1)) => ((1, 0), (1, 1)), ((-1, 3), (-1, 5)) => ((0, 2), (-1, 2)), ((3, 1), (5, 1)) => ((2, 0), (2, 1)), ((5, 1), (5, -1)) => ((2, 0), (3, 0)), ((1, 3), (1, 1)) => ((0, 1), (1, 1)), ((1, 5), (1, 3)) => ((0, 2), (1, 2)), ((3, -1), (1, -1)) => ((1, 0), (1, -1)), ((-1, 1), (-1, 3)) => ((0, 1), (-1, 1))))


In [8]:
const DIR_TO_INDEX = Dict(
    (0, -2) => 1,
    (2,  0) => 2,
    (0,  2) => 3,
    (-2, 0) => 4,
)

const possibilities = Dict{Tuple{Int, Int}, Vector{Possibility}}()  # (pento, start_orient) -> (new_orient, area_change, pento_size) 

for p in pentos
    orients = get_orientations(p.tiles)  # get all orientations (including flipping)
    # unique_orients handled by get_orientations returning unique

    for tiles in orients
        path, edge_to_tile = construct_boundary_path(tiles)
        doubled_path = vcat(path, path)  # allow wrap-around

        len_path = length(path)
        for start_index in 1:len_path
            for end_index in (start_index + 1):(start_index + len_path - 1)
                
                # denote orientation as difference vector between two endpoints of attachment edge
                p1 = doubled_path[start_index]
                p2 = doubled_path[start_index + 1]
                diff_start = (p2[1] - p1[1], p2[2] - p1[2])
                start_orient = DIR_TO_INDEX[diff_start]

                # reversed since new S is E + 1 and new S + 1 is E
                p3 = doubled_path[end_index]
                p4 = doubled_path[end_index + 1]
                diff_end = (p3[1] - p4[1], p3[2] - p4[2])
                new_orient = DIR_TO_INDEX[diff_end]

                sols = get!(possibilities, (p.id, start_orient), Possibility[])

                # extract path of inner edges translated to S + 1 at (0, 0)
                origin = doubled_path[start_index + 1]
                inner_path = [
                    (pt[1] - origin[1], pt[2] - origin[2])
                    for pt in doubled_path[(start_index + 1):end_index]
                ]

                # pre-compute area_change with shoelace formula
                area_change_8x = 0
                for e in 1:(length(inner_path) - 1)
                    area_change_8x += (
                        inner_path[e][1] * inner_path[e + 1][2]
                        - inner_path[e + 1][1] * inner_path[e][2]
                    )  # shoelace term (8x since 1/2 is missing and both coordinates are multiplied by 2x)
                end

                # calculate data for reconstruction
                start_tile_key = (doubled_path[start_index], doubled_path[start_index + 1])
                start_tile, _ = edge_to_tile[start_tile_key]  # inner tile of edge
                shifted_tiles = [(t[1] - start_tile[1], t[2] - start_tile[2]) for t in tiles]  # shift tiles such that start_tile has coordinates (0, 0)
                
                end_tile_key = (doubled_path[end_index], doubled_path[end_index + 1])
                _, next_tile = edge_to_tile[end_tile_key]  # outer tile of edge
                shifted_next_tile = (next_tile[1] - start_tile[1], next_tile[2] - start_tile[2])

                poss = Possibility(
                    new_orient,
                    area_change_8x,
                    inner_path[end],
                    p.name,
                    shifted_tiles,
                    shifted_next_tile
                )

                push!(sols, poss)  # inner_path[0] is always (0, 0), so don't need to save
            end
        end
    end
end


max_poss = maximum(length(possib) for ((_, _), possib) in possibilities)
println("No combination has more than $max_poss possibilities.")

possibilities[(1, 3)]

No combination has more than 264 possibilities.


264-element Vector{Possibility}:
 Possibility(4, 0, (0, 0), 'F', [(0, 0), (1, -1), (1, 0), (1, 1), (2, -1)], (0, 1))
 Possibility(1, 0, (2, 0), 'F', [(0, 0), (1, -1), (1, 0), (1, 1), (2, -1)], (0, 1))
 Possibility(4, 4, (2, 2), 'F', [(0, 0), (1, -1), (1, 0), (1, 1), (2, -1)], (1, 2))
 Possibility(3, 0, (4, 2), 'F', [(0, 0), (1, -1), (1, 0), (1, 1), (2, -1)], (2, 1))
 Possibility(3, -8, (4, 0), 'F', [(0, 0), (1, -1), (1, 0), (1, 1), (2, -1)], (2, 0))
 Possibility(4, -16, (4, -2), 'F', [(0, 0), (1, -1), (1, 0), (1, 1), (2, -1)], (2, 0))
 Possibility(3, -12, (6, -2), 'F', [(0, 0), (1, -1), (1, 0), (1, 1), (2, -1)], (3, -1))
 Possibility(2, -24, (6, -4), 'F', [(0, 0), (1, -1), (1, 0), (1, 1), (2, -1)], (2, -2))
 Possibility(2, -32, (4, -4), 'F', [(0, 0), (1, -1), (1, 0), (1, 1), (2, -1)], (1, -2))
 Possibility(1, -40, (2, -4), 'F', [(0, 0), (1, -1), (1, 0), (1, 1), (2, -1)], (0, -1))
 ⋮
 Possibility(4, -4, (-2, 2), 'F', [(-1, 1), (0, 0), (0, 1), (0, 2), (1, 2)], (-1, 2))
 Possibility(1, -8

In [9]:
# State packing
@inline function pack_state(pos, orient)
    # x: 12 bits, y: 12 bits, orient: 4 bits

    # shift from [-2048, 2047] to [0, 4095] to prevent negative values
    p_x = Int32(pos[1] + 2048)
    p_y = Int32(pos[2] + 2048)
    return (p_x << 16) | (p_y << 4) | Int32(orient)
end

@inline function unpack_state(packed)
    orient = Int(packed & 15)
    y = Int((packed >> 4) & 4095) - 2048
    x = Int((packed >> 16) & 4095) - 2048
    return ((x, y), orient)
end

@inline function count_set_bits(n::UInt16)
    c = 0
    while n > 0
        n &= (n - 1)
        c += 1
    end
    return c
end

@inline function get_min_set_bit(n::UInt16)
    for i in 0:11
        if ((n >> i) & 1) == 1
            return i + 1
        end
    end
    return 0
end

@inline function get_max_set_bit(n::UInt16)
    for i in 11:-1:0
        if ((n >> i) & 1) == 1
            return i + 1
        end
    end
    return 0
end

get_max_set_bit (generic function with 1 method)

In [10]:
function optimal_fence(num_pieces; find_all_solutions = false)
    # use UInt16 to store (idx << 7) | (start_orient << 4) | pento_idx
    remember = find_all_solutions ? Dict{UInt16, Dict{Int32, Vector{UInt16}}}() : Dict{UInt16, Dict{Int32, UInt16}}()
    optimal_subsets = find_all_solutions ? Set{UInt16}() : UInt16(0)

    max_fence_area = 0

    start_pos = (0, 0)
    start_orient = 3
    start = pack_state(start_pos, start_orient)

    # note 1-based, so [size + 1] corresponds to size
    table = Vector{Dict{UInt16, Dict{Int32, Int}}}(undef, num_pieces + 1)  # map set of placed pentominoes to all possible placements
    table[1] = Dict(UInt16(0) => Dict(start => 0))  # for no pentomino placed (empty set) there is only the solution that we are still at (0, 0) looking east and have 0 partial area

    start_time = time()

    for size in 1:num_pieces
        println("Starting size $size at $(round(Int, time() - start_time)) seconds. Space for size $(size - 1) is $(sum(length(v) for v in values(table[size])))")
        flush(stdout)  # ensure immediate output

        table[size+1] = Dict{UInt16, Dict{Int32, Int}}()

        for (used, solutions) in table[size]
            min_used = get_min_set_bit(used)
            max_used = get_max_set_bit(used)

            for pento_idx in 1:12
                mask = UInt16(1) << (pento_idx - 1)
                if (used & mask) != 0
                    continue  # pentomino was already placed
                end

                # eliminate rotational symmetry by enforcing that the second placed pentomino has the lowest index of all placed pentominoes
                if size == 2 && pento_idx > min_used; continue; end
                if size >= 3 && pento_idx < min_used; continue; end

                # eliminate mirror symmetry by ensuring index of third piece is larger than second piece
                if size == 3 && pento_idx < max_used; continue; end

                new_used = used | mask  # add pentomino

                # collect solutions for new used set, new_solutions is just a reference to table[size+1][new_used]
                new_solutions = get!(table[size+1], new_used) do  # do is performance improvement, since dict is only built when necessary
                    Dict{Int32, Int}()
                end 

                new_remember = get!(remember, new_used) do
                    find_all_solutions ? Dict{Int32, Vector{UInt16}}() : Dict{Int32, UInt16}()
                end

                for (packed_state, area_8x) in solutions
                    (pos, orient) = unpack_state(packed_state)
                    
                    for (idx, poss) in enumerate(possibilities[(pento_idx, orient)])
                        trans_corr = pos[2] * (-poss.pos_diff[1]) + pos[1] * (poss.pos_diff[2])  # correction for precomputed value since we have a translation from (0, 0) to pos
                        new_area_8x = area_8x + poss.area_change_8x + trans_corr

                        new_pos = pos .+ poss.pos_diff
                        
                        if size == num_pieces && !(new_pos == start_pos && poss.new_orient == start_orient)
                            continue  # all pentominoes are placed but fence is not closed
                        end

                        info = UInt16((idx << 7) | (orient << 4) | pento_idx)  # data for reconstruction
                        
                        state_key = pack_state(new_pos, poss.new_orient)

                        if !find_all_solutions
                            max_fence_area, optimal_subsets = update_solutions!(new_solutions, new_remember, state_key, new_area_8x, info, new_pos, poss.new_orient, start_pos, start_orient, size, max_fence_area, optimal_subsets, new_used)
                        else
                            max_fence_area = update_solutions_all!(new_solutions, new_remember, state_key, new_area_8x, info, new_pos, poss.new_orient, start_pos, start_orient, size, max_fence_area, optimal_subsets, new_used)
                        end
                    end
                end
            end
        end

        empty!(table[size])  # values not needed anymore
    end

    if max_fence_area > 0
        println("Found optimum in $(round(Int, time() - start_time)) seconds.")
    else
        println("No solution found.")
    end

    if !find_all_solutions
        sol_path = find_single_path(optimal_subsets, remember)
        return max_fence_area, sol_path
    end

    all_paths = Vector{Vector{ReconstructionStep}}()  # collect all solution paths

    for used in optimal_subsets
        find_all_paths!(all_paths, used, remember)  # find all paths for each optimal subset of pentominoes
    end

    return max_fence_area, all_paths
end

optimal_fence (generic function with 1 method)

In [11]:
@inline function update_solutions!(new_solutions, new_remember, state_key, new_area_8x, info, new_pos, new_orient, start_pos, start_orient, size, max_fence_area, optimal_subsets, new_used, no_output=false)
    found = haskey(new_solutions, state_key)

    if !found || new_area_8x > new_solutions[state_key]
        new_solutions[state_key] = new_area_8x
        new_remember[state_key] = info

        if new_pos == start_pos && new_orient == start_orient
            @assert new_area_8x % 8 == 0  # area must be integer

            true_area = div(new_area_8x, 8)
            if true_area > max_fence_area
                max_fence_area = true_area
                if !no_output
                    println("Size: $size, Max Fence Area: $max_fence_area")
                    flush(stdout)  # ensure immediate output
                end

                optimal_subsets = new_used
            end
        end
    end
    return max_fence_area, optimal_subsets  # since optimal_subsets is now just a number, we have to return it explicitly
end

@inline function update_solutions_all!(new_solutions, new_remember, state_key, new_area_8x, info, new_pos, new_orient, start_pos, start_orient, size, max_fence_area, optimal_subsets, new_used, no_output=false)
    found = haskey(new_solutions, state_key)

    if !found || new_area_8x >= new_solutions[state_key]  # either the first time we reach this position with those pentominoes or we improve the area
        if found && new_area_8x == new_solutions[state_key]
            push!(new_remember[state_key], info)  # equal to current best solution, collect as another predecessors
        else
            new_solutions[state_key] = new_area_8x
            new_remember[state_key] = [info]  # better than all previous values, reset collection of predecessors
        end

        if new_pos == start_pos && new_orient == start_orient
            @assert new_area_8x % 8 == 0  # area must be integer

            true_area = div(new_area_8x, 8)
            if true_area > max_fence_area
                max_fence_area = true_area
                if !no_output
                    println("Size: $size, Max Fence Area: $max_fence_area")
                    flush(stdout)  # ensure immediate output
                end

                empty!(optimal_subsets)
                push!(optimal_subsets, new_used)
            elseif true_area == max_fence_area
                push!(optimal_subsets, new_used)
            end
        end
    end
    return max_fence_area
end

update_solutions_all! (generic function with 2 methods)

In [12]:
struct ReconstructionStep
    name::Char
    tiles::Vector{Tuple{Int, Int}}
    next_tile::Tuple{Int, Int}
end

function find_single_path(used_mask, remember)
    packed_state = pack_state((0, 0), 3)
    used = used_mask

    path = Vector{ReconstructionStep}()

    while used != 0
        info = remember[used][packed_state]  # find which pentomino was used to reach current state

        # Unpack info: idx << 7 | start_orient << 4 | pento_idx
        idx = (info >> 7)
        orient = (info >> 4) & 0x07
        pento_idx = info & 0x0F

        poss = possibilities[(pento_idx, orient)][idx]
        
        # Reconstruct previous state
        (curr_pos, current_end_orient) = unpack_state(packed_state)
        prev_pos = (curr_pos[1] - poss.pos_diff[1], curr_pos[2] - poss.pos_diff[2])
        packed_state = pack_state(prev_pos, orient)

        used = used & ~(UInt16(1) << (pento_idx - 1))  # remove pento from used

        push!(path, ReconstructionStep(poss.pento_name, poss.shifted_tiles, poss.next_tile_offset))
    end

    return reverse(path)
end

function find_all_paths!(all_paths, used_mask, remember)
    # Stack: (packed_state, used, path)
    stack = Any[(pack_state((0, 0), 3), used_mask, ReconstructionStep[])]
    
    while !isempty(stack)
        (packed_state, used, path) = pop!(stack)

        if used == 0
            push!(all_paths, reverse(path))
            continue
        end
        
        for info in remember[used][packed_state]
            # Unpack info: idx << 7 | start_orient << 4 | pento_idx
            idx = (info >> 7)
            orient = (info >> 4) & 0x07
            pento_idx = info & 0x0F

            # Get possibility again
            poss = possibilities[(pento_idx, orient)][idx]
            
            # Reconstruct previous state
            (curr_pos, current_end_orient) = unpack_state(packed_state)
            # Reverse the position change
            prev_pos = (curr_pos[1] - poss.pos_diff[1], curr_pos[2] - poss.pos_diff[2])
            # The previous orientation was the start_orient of this move
            prev_packed = pack_state(prev_pos, orient)

            prev_used = used & ~(UInt16(1) << (pento_idx - 1))  # remove pento from used
            
            new_path = copy(path)
            push!(new_path, ReconstructionStep(poss.pento_name, poss.shifted_tiles, poss.next_tile_offset))
            
            push!(stack, (prev_packed, prev_used, new_path))
        end
    end
end

find_all_paths! (generic function with 1 method)

In [13]:
max_fence_area, path = optimal_fence(12)

Starting size 1 at 0 seconds. Space for size 0 is 1
Starting size 2 at 0 seconds. Space for size 1 is 730
Size: 2, Max Fence Area: 1
Starting size 3 at 0 seconds. Space for size 2 is 29519
Size: 3, Max Fence Area: 4
Size: 3, Max Fence Area: 5
Size: 3, Max Fence Area: 6
Starting size 4 at 0 seconds. Space for size 3 is 245919
Size: 4, Max Fence Area: 7
Size: 4, Max Fence Area: 9
Size: 4, Max Fence Area: 10
Size: 4, Max Fence Area: 11
Size: 4, Max Fence Area: 12
Size: 4, Max Fence Area: 13
Starting size 5 at 5 seconds. Space for size 4 is 1028126
Size: 5, Max Fence Area: 14
Size: 5, Max Fence Area: 16
Size: 5, Max Fence Area: 17
Size: 5, Max Fence Area: 18
Size: 5, Max Fence Area: 20
Size: 5, Max Fence Area: 21
Size: 5, Max Fence Area: 22
Size: 5, Max Fence Area: 23
Size: 5, Max Fence Area: 24
Starting size 6 at 20 seconds. Space for size 5 is 2621789
Size: 6, Max Fence Area: 25
Size: 6, Max Fence Area: 28
Size: 6, Max Fence Area: 29
Size: 6, Max Fence Area: 30
Size: 6, Max Fence Area: 3

(128, ReconstructionStep[ReconstructionStep('U', [(0, -1), (0, 0), (1, -1), (2, -1), (2, 0)], (3, 0)), ReconstructionStep('F', [(0, 0), (1, -1), (1, 0), (2, 0), (2, 1)], (3, 1)), ReconstructionStep('V', [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2)], (2, 3)), ReconstructionStep('X', [(-1, 1), (0, 0), (0, 1), (0, 2), (1, 1)], (0, 3)), ReconstructionStep('L', [(-1, 3), (0, 0), (0, 1), (0, 2), (0, 3)], (-1, 4)), ReconstructionStep('W', [(-2, 2), (-1, 1), (-1, 2), (0, 0), (0, 1)], (-3, 2)), ReconstructionStep('Y', [(-3, 0), (-2, 0), (-1, 0), (-1, 1), (0, 0)], (-4, 0)), ReconstructionStep('P', [(-2, -1), (-2, 0), (-1, -1), (-1, 0), (0, 0)], (-3, -1)), ReconstructionStep('Z', [(-2, -2), (-1, -2), (-1, -1), (-1, 0), (0, 0)], (-2, -3)), ReconstructionStep('I', [(0, -4), (0, -3), (0, -2), (0, -1), (0, 0)], (0, -5)), ReconstructionStep('N', [(0, -2), (0, -1), (0, 0), (1, -3), (1, -2)], (2, -3)), ReconstructionStep('T', [(0, -2), (0, -1), (0, 0), (1, -1), (2, -1)], (3, -1))])

In [13]:
function process_path(path)
    fence_tiles = Dict{Tuple{Int, Int}, Char}()
    curr = (0, 0)
    
    for step in path
        for (x, y) in step.tiles
            @assert !haskey(fence_tiles, (x + curr[1], y + curr[2]))  # check for self-Intersecting pentominoes
            fence_tiles[(x + curr[1], y + curr[2])] = step.name
        end
        curr = (curr[1] + step.next_tile[1], curr[2] + step.next_tile[2])  # update current coordinates 
    end
    return fence_tiles
end

# Print the grid visualization
function print_grid(fence_tiles; io::IO = stdout)
    if isempty(fence_tiles)
        println("Empty grid")
        return
    end
    
    xs = [k[1] for k in keys(fence_tiles)]
    ys = [k[2] for k in keys(fence_tiles)]
    min_x, max_x = minimum(xs), maximum(xs)
    min_y, max_y = minimum(ys), maximum(ys)
    
    grid = [['.' for _ in min_x:max_x] for _ in min_y:max_y]
    
    for ((x, y), c) in fence_tiles
        grid[max_y - y + 1][x - min_x + 1] = c
    end
    
    for row in grid
        println(io, join(row))
    end
end

function calculate_enclosed_areas(fence_tiles)
    if isempty(fence_tiles)
        return Int[]
    end

    xs = [k[1] for k in keys(fence_tiles)]
    ys = [k[2] for k in keys(fence_tiles)]
    min_x, max_x = minimum(xs), maximum(xs)
    min_y, max_y = minimum(ys), maximum(ys)
    
    areas = Int[]
    visited = Set{Tuple{Int, Int}}()  # Track visited cells
    dirs8 = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]  # 8-connectivity: all 8 directions including diagonals
    
    for x in min_x:max_x
        for y in min_y:max_y
            if !haskey(fence_tiles, (x, y)) && !((x, y) in visited)
                # BFS search for all connected components in the complement that are not connected to the outside
                push!(visited, (x, y))
                q = [(x, y)]
                size = 1
                touches_border = false
                
                head = 1
                while head <= length(q)
                    cx, cy = q[head]
                    head += 1
                    
                    if cx == min_x || cx == max_x || cy == min_y || cy == max_y
                        touches_border = true
                    end
                    
                    for (dx, dy) in dirs8
                        nx, ny = cx + dx, cy + dy
                        if nx >= min_x && nx <= max_x && ny >= min_y && ny <= max_y
                            if !haskey(fence_tiles, (nx, ny)) && !((nx, ny) in visited)
                                push!(visited, (nx, ny))
                                push!(q, (nx, ny))
                                size += 1
                            end
                        end
                    end
                end
                if !touches_border
                    push!(areas, size)
                end
            end
        end
    end
    sort!(areas, rev=true)  # sort in descending order
    return areas
end

calculate_enclosed_areas (generic function with 1 method)

In [15]:
fence_tiles = process_path(path)
print_grid(fence_tiles)

areas = calculate_enclosed_areas(fence_tiles)
@assert length(areas) == 1 && areas[1] == max_fence_area  # ensure area matches maximum

........Y......
...PPPYYYYWW...
.ZZPP......WW..
.Z..........W..
ZZ..........LL.
I............L.
I............L.
I............L.
I............X.
I...........XXX
N............X.
N............V.
NN...........V.
.NT.......FVVV.
..TTTU.UFFF....
..T..UUU.F.....


In [16]:
max_fence_area, all_paths = optimal_fence(12, find_all_solutions = true)
println("Total number of paths found: $(length(all_paths))\n")

Starting size 1 at 0 seconds. Space for size 0 is 1
Starting size 2 at 0 seconds. Space for size 1 is 730
Size: 2, Max Fence Area: 1
Starting size 3 at 0 seconds. Space for size 2 is 29519
Size: 3, Max Fence Area: 4
Size: 3, Max Fence Area: 5
Size: 3, Max Fence Area: 6
Starting size 4 at 0 seconds. Space for size 3 is 245919
Size: 4, Max Fence Area: 7
Size: 4, Max Fence Area: 9
Size: 4, Max Fence Area: 10
Size: 4, Max Fence Area: 11
Size: 4, Max Fence Area: 12
Size: 4, Max Fence Area: 13
Starting size 5 at 5 seconds. Space for size 4 is 1028126
Size: 5, Max Fence Area: 14
Size: 5, Max Fence Area: 16
Size: 5, Max Fence Area: 17
Size: 5, Max Fence Area: 18
Size: 5, Max Fence Area: 20
Size: 5, Max Fence Area: 21
Size: 5, Max Fence Area: 22
Size: 5, Max Fence Area: 23
Size: 5, Max Fence Area: 24
Starting size 6 at 21 seconds. Space for size 5 is 2621789
Size: 6, Max Fence Area: 25
Size: 6, Max Fence Area: 28
Size: 6, Max Fence Area: 29
Size: 6, Max Fence Area: 30
Size: 6, Max Fence Area: 3

In [21]:
mkpath("Results")  # ensure folder exists

output_file = "Results/all-sols.txt"

open(output_file, "w") do f
    for (idx, path) in enumerate(all_paths)
        println(f, "Solution $(idx):")

        fence_tiles = process_path(path)
        print_grid(fence_tiles; io=f)

        areas = calculate_enclosed_areas(fence_tiles)
        @assert length(areas) == 1 && areas[1] == max_fence_area 
        println(f, "")
    end
end

println("All solutions saved to $output_file")

All solutions saved to Results/all-sols.txt


In [ ]:
function optimal_order_fence!(order, table = nothing, remember = nothing; find_all_solutions = false)
    num_pieces = length(order)

    # Precompute masks for all prefixes
    masks = Vector{UInt16}(undef, num_pieces + 1)
    masks[1] = UInt16(0)
    current_mask = UInt16(0)
    for i in 1:num_pieces
        current_mask |= (UInt16(1) << (order[i] - 1))
        masks[i+1] = current_mask
    end

    if remember === nothing
        # use UInt16 to store (idx << 7) | (start_orient << 4) | pento_idx
        remember = find_all_solutions ? Dict{UInt16, Dict{Int32, Vector{UInt16}}}() : Dict{UInt16, Dict{Int32, UInt16}}()
    end

    optimal_subsets = find_all_solutions ? Set{UInt16}() : UInt16(0)  # not strictly necessary, but kept for compatibility 

    max_fence_area = 0

    start_pos = (0, 0)
    start_orient = 3
    start = pack_state(start_pos, start_orient)

    start_index = 0

    if table === nothing || length(table) != num_pieces + 1
        table = Vector{Dict{UInt16, Dict{Int32, Int}}}(undef, num_pieces + 1)  # map set of placed pentominoes to all possible placements
    else
        # findest longest prefix of 'order' that is already computed
        while start_index < num_pieces - 1 && haskey(table[start_index + 2], masks[start_index + 2])
            start_index += 1
        end
    end

    if start_index == 0
        table[1] = Dict(UInt16(0) => Dict(start => 0))  # for no pentomino placed (empty set) there is only the solution that we are still at (0, 0) looking east and have 0 partial area
    end

    for size in (start_index + 1):num_pieces
        table[size+1] = Dict{UInt16, Dict{Int32, Int}}()
        
        pento_idx = order[size]

        for (used, solutions) in table[size]
            new_used = masks[size+1]  # simply choose order prefix

            # collect solutions for new used set, new_solutions is just a reference to table[size+1][new_used]
            new_solutions = get!(table[size+1], new_used) do  # do is performance improvement, since dict is only built when necessary
                Dict{Int32, Int}()
            end 

            new_remember = get!(remember, new_used) do
                find_all_solutions ? Dict{Int32, Vector{UInt16}}() : Dict{Int32, UInt16}()
            end

            for (packed_state, area_8x) in solutions
                (pos, orient) = unpack_state(packed_state)
                
                for (idx, poss) in enumerate(possibilities[(pento_idx, orient)])
                    trans_corr = pos[2] * (-poss.pos_diff[1]) + pos[1] * (poss.pos_diff[2])  # correction for precomputed value since we have a translation from (0, 0) to pos
                    new_area_8x = area_8x + poss.area_change_8x + trans_corr

                    new_pos = pos .+ poss.pos_diff
                    
                    if size == num_pieces && !(new_pos == start_pos && poss.new_orient == start_orient)
                        continue  # all pentominoes are placed but fence is not closed
                    end

                    info = UInt16((idx << 7) | (orient << 4) | pento_idx)  # data for reconstruction
                    
                    state_key = pack_state(new_pos, poss.new_orient)

                    if !find_all_solutions
                        max_fence_area, optimal_subsets = update_solutions!(new_solutions, new_remember, state_key, new_area_8x, info, new_pos, poss.new_orient, start_pos, start_orient, size, max_fence_area, optimal_subsets, new_used, true)
                    else
                        max_fence_area = update_solutions_all!(new_solutions, new_remember, state_key, new_area_8x, info, new_pos, poss.new_orient, start_pos, start_orient, size, max_fence_area, optimal_subsets, new_used, true)
                    end
                end
            end
        end
    end

    if max_fence_area == 0
        println("No solution found.")
        return
    end

    if !find_all_solutions
        sol_path = find_single_path(optimal_subsets, remember)
        return max_fence_area, sol_path, table, remember
    end

    all_paths = Vector{Vector{ReconstructionStep}}()  # collect all solution paths

    for used in optimal_subsets
        find_all_paths!(all_paths, used, remember)  # find all paths for each optimal subset of pentominoes
    end

    return max_fence_area, all_paths, table, remember
end

optimal_order_fence! (generic function with 3 methods)

In [15]:
order = [8,2,11,12,9,5,4,10,1,6,7,3]

max_fence_area, path, _, _ = optimal_order_fence!(order)
fence_tiles = process_path(path)

areas = calculate_enclosed_areas(fence_tiles)
@assert length(areas) == 1 && areas[1] == max_fence_area
println("Order $order has maximum area $max_fence_area\n")

print_grid(fence_tiles)

Order [8, 2, 11, 12, 9, 5, 4, 10, 1, 6, 7, 3] has maximum area 116

......W.......
...PPPWW......
..NPP..WWZ....
.NN......ZZZ.Y
.N.........ZYY
.N...........Y
.X...........Y
XXX..........I
.X...........I
.F...........I
FF...........I
.FF..........I
..T..........V
..T..........V
.TTTU.U...LVVV
....UUULLLL...


In [17]:
order = [1,10,3,12,5,2,6,9,4,11,7,8]

max_fence_area, all_paths, _, _ = optimal_order_fence!(order; find_all_solutions = true)
println("Order $order has maximum area $max_fence_area")
println("Total number of solutions found: $(length(all_paths))\n")

for (idx, path) in enumerate(all_paths)
    println("Solution $(idx):")

    fence_tiles = process_path(path)
    print_grid(fence_tiles)

    areas = calculate_enclosed_areas(fence_tiles)
    @assert length(areas) == 1 && areas[1] == max_fence_area 
    println("")
end

Order [1, 10, 3, 12, 5, 2, 6, 9, 4, 11, 7, 8] has maximum area 111
Total number of solutions found: 15

Solution 1:
..T............
..TTTIIIIIPPP..
..T........PP..
.WW.........ZZ.
WW...........Z.
W............ZZ
N.............L
N.............L
NN............L
.N...........LL
.Y...........X.
.Y......V..FXXX
.YYU.U..VFFF.X.
.Y.UUUVVV.F....

Solution 2:
..T............
..TTTIIIIIPPP..
..T........PP..
.WW.........Z..
WW..........ZZZ
W.............Z
N.............L
N.............L
NN............L
.N...........LL
.Y...........X.
.Y......V..FXXX
.YYU.U..VFFF.X.
.Y.UUUVVV.F....

Solution 3:
.......T.......
...WWTTT.......
..WW...TIIIII..
..W.........PP.
..N.........PP.
.NN..........P.
.N...........ZZ
.N...........Z.
.Y..........ZZ.
.Y..........L..
YY..........L..
.Y..........L..
.U.U.......LL..
.UUUV......X...
....V....FXXX..
....VVVFFF.X...
........F......

Solution 4:
........T.......
....WWTTT.......
...WW...TIIIII..
..NW.........PP.
.NN..........PP.
.N............P.
.N............ZZ
.Y....